In [ ]:
import wget

!curl -o input.txt https://raw.githubusercontent.com/karpathy/ng-video-lecture/master/input.txt

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1089k  100 1089k    0     0  1353k      0 --:--:-- --:--:-- --:--:-- 1353k


## TOKENIZER

In [44]:
raw_text = open("input.txt", "r").read()
print(raw_text[:1000])  

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



#### Create tokens

In [31]:
import re

preprocessed = re.split(r'([.,;:?_!&"()\']|--|\s)', raw_text)

preprocessed = [token.strip() for token in preprocessed if token.strip()]
print(len(preprocessed))
print(preprocessed[:100])


260470
['First', 'Citizen', ':', 'Before', 'we', 'proceed', 'any', 'further', ',', 'hear', 'me', 'speak', '.', 'All', ':', 'Speak', ',', 'speak', '.', 'First', 'Citizen', ':', 'You', 'are', 'all', 'resolved', 'rather', 'to', 'die', 'than', 'to', 'famish', '?', 'All', ':', 'Resolved', '.', 'resolved', '.', 'First', 'Citizen', ':', 'First', ',', 'you', 'know', 'Caius', 'Marcius', 'is', 'chief', 'enemy', 'to', 'the', 'people', '.', 'All', ':', 'We', 'know', "'", 't', ',', 'we', 'know', "'", 't', '.', 'First', 'Citizen', ':', 'Let', 'us', 'kill', 'him', ',', 'and', 'we', "'", 'll', 'have', 'corn', 'at', 'our', 'own', 'price', '.', 'Is', "'", 't', 'a', 'verdict', '?', 'All', ':', 'No', 'more', 'talking', 'on', "'", 't']


#### Create token IDs

In [ ]:
all_words = sorted(set(preprocessed))
vocab_size = len(all_words)

print(vocab_size)
print(all_words[:100])

vocab = {token:integer for integer, token in enumerate(all_words)}

for x in vocab.items():
    print(x)

13852
['!', '&', "'", ',', '--', '.', '3', ':', ';', '?', 'A', 'ABHORSON', 'ABRAHAM', 'ADRIAN', 'AEacides', 'AEdile', 'AEdiles', 'AEneas', 'AEsop', 'ALL', 'ALONSO', 'ANGELO', 'ANNE', 'ANOTHER', 'ANTIGONUS', 'ANTONIO', 'ARCHBISHOP', 'ARCHIDAMUS', 'ARIEL', 'AUFIDIUS', 'AUMERLE', 'AUTOLYCUS', 'Abase', 'Abate', 'Abated', 'Abbot', 'Abel', 'Abhorred', 'Abhorson', 'Abides', 'Able', 'About', 'Above', 'Abraham', 'Absolute', 'Accept', 'Accomplish', 'According', 'Accords', 'Account', 'Accountant', 'Accursed', 'Accuse', 'Achieve', 'Acquaint', 'Action', 'Adam', 'Add', 'Added', 'Adding', 'Address', 'Adieu', 'Adjudged', 'Admit', 'Adonis', 'Adoptedly', 'Adopts', 'Adrian', 'Adriatic', 'Advance', 'Advantaging', 'Adversity', 'Advertising', 'Advocate', 'Affection', 'Affliction', 'Affrighted', 'Affrights', 'Affront', 'Afore', 'Afresh', 'Afric', 'African', 'After', 'Again', 'Against', 'Agamemnon', 'Age', 'Aged', 'Agenor', 'Agreed', 'Agrippa', 'Ah', 'Aim', 'Aiming', 'Airy', 'Ajax', 'Al', 'Alack', 'Alas']
('!

In [ ]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab                             #token to integer vocab
        self.int_to_str = {i:s for s,i in vocab.items()}    #integer to token vocab

    def encode(self, text):
        preprocessed = re.split(r'([.,;:?_!&"()\']|--|\s)', text)
        preprocessed = [token.strip() for token in preprocessed if token.strip()]

        return [self.str_to_int[token] for token in preprocessed]

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)  #removes spaces before punctuation
        return text

tokenizer = SimpleTokenizerV1(vocab)

text = """First Citizen:
First, you know Caius Marcius is chief enemy to the people.
""" 

ids = tokenizer.encode(text)
print(ids)

print(tokenizer.decode(ids))

[934, 492, 7, 934, 3, 13835, 8101, 424, 1543, 7939, 4421, 5969, 12581, 12408, 9676, 5]
First Citizen : First, you know Caius Marcius is chief enemy to the people.


##### Adding new tokens for unkown words and end of the text

In [55]:
all_tokens = sorted(list(set(preprocessed)))
print(len(all_tokens))

all_tokens.extend(["<|unk|>", "<|endoftext|>"])

vocab = {token:integer for integer, token in enumerate(all_tokens)}

13852


In [58]:
print(len(vocab))

for item in list(vocab.items())[-5:]:
    print(item)

13854
('zenith', 13849)
('zodiacs', 13850)
('zounds', 13851)
('<|unk|>', 13852)
('<|endoftext|>', 13853)


In [61]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}

    def encode(self, text):
        preprocessed = re.split(r'([.,;:?_!&"()\']|--|\s)', text)
        preprocessed = [token.strip() for token in preprocessed if token.strip()]
        preprocessed = [item if item in self.str_to_int else "<|unk|>" for item in preprocessed]
        return [self.str_to_int[token] for token in preprocessed]

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

tokenizer = SimpleTokenizerV2(vocab)

text = """Hello, how are you?"""
print(tokenizer.encode(text))

tokenizer.decode(tokenizer.encode(text))

[13852, 3, 7582, 3274, 13835, 9]


'<|unk|>, how are you?'

### BYTE PAIR Encoding -> tokens that aren't predefined

In [66]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.n_vocab)
print(tokenizer)

50257
<Encoding 'gpt2'>


In [104]:
text= (
    "Hello, dou you like tea? <|endoftext|> In the sunlit terraces of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)

[15496, 11, 2255, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 286, 617, 34680, 27271, 13]


### Creating INPUT-TARGET Pairs

In [79]:
encoded_text = tokenizer.encode(raw_text)
print(len(encoded_text))

338025


## Lecture 10: Token Embeddings

In [ ]:
import torch
input_ids = torch.tensor([2,3,5,1])
print(input_ids.shape)

vocab_size = 6 #number of tokens in the vocabulary
output_dim = 3 #embedding dimension

torch.manual_seed(123)

#embeddings_layer = a lookup matrix that accept as parameter a token id and return its embedding
embeddings_layer = torch.nn.Embedd ing(vocab_size, output_dim)
print(embeddings_layer.weight)


print(embeddings_layer(torch.tensor([0])))  #goes to the lookup matrix and returns the embedding for the token id = 0 ( the first row in the lookup matrix)
print(type(torch.tensor([0])))
print(embeddings_layer(input_ids))



torch.Size([4])
Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)
tensor([[ 0.3374, -0.1778, -0.1690]], grad_fn=<EmbeddingBackward0>)
<class 'torch.Tensor'>
tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


## L11: Positional Embeddings - Encoding Word Positions

In [ ]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
 

Parameter containing:
tensor([[ 1.4424,  2.6252, -0.0923,  ...,  1.9454, -1.1768,  0.8824],
        [-0.4691,  1.2547, -1.3212,  ..., -1.5919, -0.0203, -0.6094],
        [ 0.2324, -0.9103, -0.4608,  ...,  1.5701, -0.2833, -0.4178],
        ...,
        [ 0.4721,  0.4064, -2.4622,  ..., -0.2801, -1.0035, -0.2706],
        [-0.0434,  0.6163, -0.3900,  ..., -0.6484, -0.2449,  1.6638],
        [ 0.6332, -1.2952,  0.4919,  ...,  0.8049,  0.6275, -0.0446]],
       requires_grad=True)


In [ ]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size = 8, max_length = max_length,
    stride = max_length
)